# SARAI - Modèle 1 : Keyword Search Chatbot

**Stocktaking of Arab Regional AI Initiatives**

Chatbot basé sur la recherche par mots-clés (ILIKE) dans PostgreSQL/SQLite.

Ce notebook sert de **baseline** pour les modèles :
- Modèle 2 : Semantic Search (BGE + ChromaDB)
- Modèle 3 : RAG (BGE + ChromaDB + Mistral 7B)

---
## Architecture

```
Question → Query Parser (stop words, intent, pays, secteur, tech)
         → ILIKE Search (OR entre mots-clés sur toutes les colonnes)
         → Scoring (proportion de mots-clés matchés)
         → Build Response (FR/EN)
```

In [1]:
# ── Imports ──
import sys, os, re, json, time, warnings
from urllib.parse import quote_plus
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
warnings.filterwarnings('ignore')

print('Imports OK')

Imports OK


In [2]:
# ── Connexion DB (PostgreSQL -> fallback SQLite) ──

# Chercher la base dans backend/ depuis le repertoire courant
cwd = os.getcwd()
candidates = [
    os.path.join(cwd, '..', 'backend'),          # chatbot/../backend
    os.path.join(cwd, 'backend'),                # si lance depuis la racine
    os.path.abspath('../backend'),
    os.path.abspath('backend'),
]

backend_dir = None
for d in candidates:
    d = os.path.abspath(d)
    if os.path.exists(os.path.join(d, 'sarai.db')):
        backend_dir = d
        break

if backend_dir is None:
    raise FileNotFoundError(
        "Base de donnees introuvable. Cherche dans: "
        + ", ".join(os.path.abspath(d) for d in candidates)
    )

print(f'[DB] Backend trouve: {backend_dir}')

env_path = os.path.join(backend_dir, '.env')
if os.path.exists(env_path):
    load_dotenv(env_path)

password = os.getenv('DB_PASSWORD', '0000')
db_user = os.getenv('DB_USER', 'postgres')
db_host = os.getenv('DB_HOST', 'localhost')
db_port = os.getenv('DB_PORT', '5432')
db_name = os.getenv('DB_NAME', 'SARAI_DB')

SQLITE_PATH = os.path.join(backend_dir, 'sarai.db')
SQLITE_URL = f'sqlite:///{SQLITE_PATH}'

engine = None
if db_host:
    DATABASE_URL = f'postgresql+psycopg://{db_user}:{quote_plus(password)}@{db_host}:{db_port}/{db_name}'
    try:
        test_engine = create_engine(DATABASE_URL, pool_pre_ping=True)
        with test_engine.connect() as conn:
            conn.execute(text('SELECT 1'))
        engine = create_engine(DATABASE_URL, pool_pre_ping=True)
        print(f'[DB] PostgreSQL OK: {db_host}:{db_port}/{db_name}')
    except Exception as e:
        print(f'[DB] PostgreSQL indisponible: {e}')

if engine is None:
    print(f'[DB] SQLite: {SQLITE_PATH}')
    engine = create_engine(SQLITE_URL, connect_args={'check_same_thread': False})

SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)
db = SessionLocal()

# Verifier les donnees
for t in ['projects', 'stakeholders', 'resources']:
    try:
        c = db.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'[DB] {t}: {c} lignes')
    except Exception as ex:
        print(f'[DB] {t}: ABSENTE - {ex}')

# Apercu
try:
    row = db.execute(text('SELECT p.id, p.title, p.sector, c.country FROM projects p LEFT JOIN countries c ON c.id=p.country_id LIMIT 5')).fetchall()
    print('\nApercu projets:')
    for r in row:
        print(f'  {r.id}: {r.title} | {r.sector} | {r.country}')
except Exception as ex:
    print(f'Apercu non disponible: {ex}')

[DB] Backend trouve: c:\Users\user\OneDrive - ESPRIT\Documents\GitHub\test\Stage-PFE-AICTO\backend
[DB] PostgreSQL indisponible: No module named 'psycopg'
[DB] SQLite: c:\Users\user\OneDrive - ESPRIT\Documents\GitHub\test\Stage-PFE-AICTO\backend\sarai.db
[DB] projects: 12 lignes
[DB] stakeholders: 12 lignes
[DB] resources: 5 lignes

Apercu projets:
  1: AI Diagnostic System | Health | Egypt
  2: Smart Irrigation System | Agriculture | Morocco
  3: Adaptive Learning Platform | Education | UAE
  4: Traffic Management AI | Transportation | Saudi Arabia
  5: Financial Fraud Detection | Finance | UAE


---
## 1. Query Parser

Nettoie la question, extrait les mots-clés (sans les mots d'intention ni les stop words), détecte le pays, le type d'entité, le secteur et la technologie.

In [4]:
# ── Dictionnaires ──

INTENT_WORDS = {
    'project','projects','projet','projets','initiative','initiatives',
    'stakeholder','stakeholders','organization','organizations','organisation','organisations',
    'resource','resources','ressource','ressources',
    'startup','startups','ngo','ngos','company','companies','lab','labs','center','centre',
    'report','reports','dataset','datasets','publication','publications','tool','tools','guide','guides',
    'show','shows','list','liste','give','gives','find','finds','search',
    'montre','montrer','moi','donne','donner','lister','affiche','afficher',
    'cherche','chercher','trouve','trouver','veux','peux','peut',
    'all','tous','toutes','les','des',
    'quels','quelles','quel','quelle','me','you','your','my',
}

STOP_WORDS = {
    'le','la','les','des','de','du','un','une','et','est','sont',
    'dans','pour','sur','avec','par','pas','que','qui','quoi',
    'the','a','an','in','on','at','to','for','of','and','or',
    'is','are','was','were','be','been','being','have','has',
    'had','do','does','did','will','would','could','should',
    'may','might','shall','can','need','dare','ought','used',
    'what','which','who','whom','this','that','these','those',
    'am','it','its','some','any','each','every','all','both',
    'few','more','most','other','such','no','nor','not','only',
    'own','same','so','than','too','very','just','because','as',
    'until','while','if','else','when','where','why','how',
    'about','between','through','during','before','after','above',
    'below','up','down','out','off','over','under','again',
    'further','then','once','here','there','en','y','a','au','aux',
    'avoir','etre','faire',
}

COUNTRY_MAP = {
    'tunisia':'Tunisia','tunisie':'Tunisia','algeria':'Algeria','algerie':'Algeria',
    'morocco':'Morocco','maroc':'Morocco','egypt':'Egypt','egypte':'Egypt',
    'uae':'UAE','emirates':'UAE','dubai':'UAE','saudi':'Saudi Arabia',
    'qatar':'Qatar','kuwait':'Kuwait','oman':'Oman','bahrain':'Bahrain',
    'lebanon':'Lebanon','liban':'Lebanon','jordan':'Jordan','iraq':'Iraq',
    'yemen':'Yemen','syria':'Syria','palestine':'Palestine',
    'mauritania':'Mauritania','libya':'Libya','sudan':'Sudan','somalia':'Somalia',
    'djibouti':'Djibouti','comoros':'Comoros',
}

ENTITY_MAP = {
    'project': {'project','projects','projet','projets','initiative','initiatives'},
    'stakeholder': {'stakeholder','stakeholders','organization','organizations',
                    'organisation','organisations','startup','startups',
                    'ngo','ngos','company','companies','lab','labs'},
    'resource': {'resource','resources','ressource','ressources',
                 'report','reports','dataset','datasets'},
}

SECTOR_MAP = {
    'health': {'health','healthcare','medical','sante','hospital','telemedicine'},
    'education': {'education','school','university','training','learning','edutech','adaptive'},
    'agriculture': {'agriculture','agricultural','farming','food','agritech','irrigation'},
    'finance': {'finance','financial','banking','fintech','fraud'},
    'energy': {'energy','renewable','solar','wind','power'},
    'environment': {'environment','environmental','climate','water','green','waste'},
    'transportation': {'transportation','transport','traffic','logistics'},
    'security': {'security','cyber','cybersecurity'},
}

TECH_MAP = {
    'nlp':'NLP','natural language':'NLP','llm':'NLP',
    'computer vision':'Computer Vision','vision':'Computer Vision',
    'machine learning':'Machine Learning','ml':'Machine Learning',
    'deep learning':'Deep Learning','dl':'Deep Learning',
    'robotics':'Robotics','robot':'Robotics','blockchain':'Blockchain',
    'iot':'IoT','internet of things':'IoT','big data':'Big Data',
    'generative':'Generative AI','genai':'Generative AI',
    'speech recognition':'Speech Recognition','speech':'Speech Recognition',
}

print(f'Dictionnaires charges: {len(COUNTRY_MAP)} pays, {len(ENTITY_MAP)} entites, {len(SECTOR_MAP)} secteurs, {len(TECH_MAP)} technologies')

Dictionnaires charges: 29 pays, 3 entites, 8 secteurs, 19 technologies


In [5]:
# ── Fonctions de parsing ──

def extract_keywords(query):
    """Extrait les mots-cles significatifs (filtre stop words + mots d'intention)."""
    lower = query.lower().strip().replace('-',' ').replace("'",' ').replace('_',' ')
    tokens = re.split(r"[\s,;:!?()]+", lower)
    return [t for t in tokens if len(t) > 1 
            and t not in STOP_WORDS 
            and t not in INTENT_WORDS]

def detect_country(query):
    lower = query.lower()
    for alias, country in COUNTRY_MAP.items():
        if alias in lower:
            return country
    return None

def detect_entity(query):
    lower = query.lower()
    for entity, keywords in ENTITY_MAP.items():
        for kw in keywords:
            if kw in lower:
                return entity
    return None

def detect_sector(query):
    lower = query.lower()
    for sector, keywords in SECTOR_MAP.items():
        for kw in keywords:
            if kw in lower:
                return sector.capitalize()
    return None

def detect_tech(query):
    lower = query.lower()
    for alias, tech in TECH_MAP.items():
        if alias in lower:
            return tech
    return None

def detect_lang(query):
    french = {'quels','quelles','quel','quelle','projets','sante','tunisie',
              'maroc','donne','montre','sante'}
    tokens = set(re.split(r"[\s,;:!?()]+", query.lower().strip()))
    arabic_chars = set('ابتثجحخدذرزسشصضطظعغفقكلمنهويآأؤإئ')
    if any(c in query for c in arabic_chars):
        return 'ar'
    if tokens & french:
        return 'fr'
    return 'en'

def parse_query(query):
    return {
        'original': query,
        'keywords': extract_keywords(query),
        'country': detect_country(query),
        'entity': detect_entity(query),
        'sector': detect_sector(query),
        'technology': detect_tech(query),
        'language': detect_lang(query),
    }

# Test du parser
test_queries = [
    'Quels projets IA en sante en Tunisie ?',
    'NLP projects in Morocco',
    'Show me AI startups in Egypt',
    'Machine learning resources for healthcare',
    'AI healthcare projects',
    'Arabic speech recognition',
]
print('Test du Query Parser:')
print('='*70)
for q in test_queries:
    p = parse_query(q)
    print(f'\n{q}')
    print(f'  keywords={p["keywords"]}')
    print(f'  pays={p["country"]} | type={p["entity"]} | secteur={p["sector"]} | tech={p["technology"]} | lang={p["language"]}')

Test du Query Parser:

Quels projets IA en sante en Tunisie ?
  keywords=['ia', 'sante', 'tunisie']
  pays=Tunisia | type=project | secteur=Health | tech=None | lang=fr

NLP projects in Morocco
  keywords=['nlp', 'morocco']
  pays=Morocco | type=project | secteur=None | tech=NLP | lang=en

Show me AI startups in Egypt
  keywords=['ai', 'egypt']
  pays=Egypt | type=stakeholder | secteur=None | tech=None | lang=en

Machine learning resources for healthcare
  keywords=['machine', 'learning', 'healthcare']
  pays=None | type=resource | secteur=Health | tech=Machine Learning | lang=en

AI healthcare projects
  keywords=['ai', 'healthcare']
  pays=None | type=project | secteur=Health | tech=None | lang=en

Arabic speech recognition
  keywords=['arabic', 'speech', 'recognition']
  pays=None | type=None | secteur=None | tech=Speech Recognition | lang=en


---
## 2. Moteur de recherche ILIKE

Recherche dans les 3 tables avec **OR entre mots-clés** (n'importe quel mot-clé peut matcher n'importe quelle colonne).
Le score = `(mots matchés) / (total mots)` avec bonus si tous les mots matchent.

In [6]:
# ── Fonctions de recherche ──

def build_ilike_conditions(params, alias, columns, keywords, prefix='w'):
    """Construit OR entre toutes les colonnes × tous les mots-clés."""
    if not keywords:
        return '1=1', params
    clauses = []
    for i, kw in enumerate(keywords):
        for j, col in enumerate(columns):
            pname = f'{prefix}_{i}_{j}'
            clauses.append(f"LOWER({alias}.{col}) LIKE :{pname}")
            params[pname] = f'%{kw}%'
    return '(' + ' OR '.join(clauses) + ')', params

def count_matches(keywords, *text_fields):
    """Compte combien de mots-cles matchent dans les champs texte."""
    if not keywords:
        return 1
    return sum(1 for kw in keywords 
               if any(kw in (t or '').lower() for t in text_fields))

def search_projects(s, keywords, country=None, sector=None, tech=None, limit=10):
    if not keywords and not country and not sector and not tech:
        return []
    params = {}
    where = []
    
    # Condition mots-cles (OR entre tous les mots × colonnes)
    if keywords:
        cond, params = build_ilike_conditions(params, 'p', 
            ['title','description','sector','technology'], keywords)
        where.append(cond)
    
    # Filtres
    if country:
        where.append('LOWER(c.country) LIKE :country')
        params['country'] = f'%{country.lower()}%'
    if sector:
        where.append('(LOWER(p.sector) = :sector OR LOWER(p.sector) LIKE :sector_like)')
        params['sector'] = sector.lower()
        params['sector_like'] = f'%{sector.lower()}%'
    if tech:
        where.append('(LOWER(p.technology) = :tech OR LOWER(p.technology) LIKE :tech_like)')
        params['tech'] = tech.lower()
        params['tech_like'] = f'%{tech.lower()}%'
    
    where.append("p.status NOT IN ('pending','rejected')")
    
    sql = f'''
        SELECT p.id,p.title,p.description,p.sector,p.technology,
               p.year_of_implementation,p.organization,
               c.country AS country
        FROM projects p
        LEFT JOIN countries c ON c.id = p.country_id
        WHERE {' AND '.join(where)}
        ORDER BY p.updated_at DESC
        LIMIT :lim
    '''
    params['lim'] = limit
    rows = s.execute(text(sql), params).fetchall()
    n = max(len(keywords), 1)
    return [{
        'id': r.id,
        'title': r.title,
        'type': 'project',
        'country': r.country or '',
        'sector': r.sector or '',
        'technology': r.technology or '',
        'description': (r.description or '')[:200],
        'score': min(1.0, count_matches(keywords, r.title, r.description, r.sector, r.technology) / n
                     + (0.2 if count_matches(keywords, r.title, r.description, r.sector, r.technology) == n and keywords else 0)),
    } for r in rows]

def search_stakeholders(s, keywords, country=None, limit=10):
    if not keywords and not country:
        return []
    params = {}
    where = []
    if keywords:
        cond, params = build_ilike_conditions(params, 's',
            ['name','description','type','country'], keywords, 'sw')
        where.append(cond)
    if country:
        where.append('LOWER(s.country) LIKE :country')
        params['country'] = f'%{country.lower()}%'
    sql = f'''
        SELECT s.id,s.name AS title,s.description,s.type,s.country,s.category
        FROM stakeholders s
        WHERE {' AND '.join(where)}
        ORDER BY s.updated_at DESC
        LIMIT :lim
    '''
    params['lim'] = limit
    rows = s.execute(text(sql), params).fetchall()
    n = max(len(keywords), 1)
    return [{
        'id': r.id,
        'title': r.title,
        'type': 'stakeholder',
        'country': r.country or '',
        'sector': r.type or '',
        'description': (r.description or '')[:200],
        'score': count_matches(keywords, r.title, r.description, r.type, r.country) / n if keywords else 0.5,
    } for r in rows]

def search_resources(s, keywords, limit=10):
    if not keywords:
        return []
    params = {}
    cond, params = build_ilike_conditions(params, 'r',
        ['title','description','category','type'], keywords, 'rw')
    sql = f'''
        SELECT r.id,r.title,r.description,r.type,r.category,r.language
        FROM resources r
        WHERE {cond}
        ORDER BY r.updated_at DESC
        LIMIT :lim
    '''
    params['lim'] = limit
    rows = s.execute(text(sql), params).fetchall()
    n = max(len(keywords), 1)
    return [{
        'id': r.id,
        'title': r.title,
        'type': 'resource',
        'country': '',
        'sector': r.category or '',
        'description': (r.description or '')[:200],
        'score': count_matches(keywords, r.title, r.description, r.category, r.type) / n,
    } for r in rows]

print('Fonctions de recherche chargees')

Fonctions de recherche chargees


In [7]:
# ── Build Response ──

def build_response(items, lang='en'):
    """Construit une reponse textuelle lisible."""
    if not items:
        return {
            'fr': "Aucun resultat trouve. Essayez avec d'autres mots-cles.",
            'en': 'No results found. Try different keywords.',
            'ar': '\u0644\u0645 \u064a\u062a\u0645 \u0627\u0644\u0639\u062b\u0648\u0631 \u0639\u0644\u0649 \u0646\u062a\u0627\u0626\u062c. \u062d\u0627\u0648\u0644 \u0628\u0643\u0644\u0645\u0627\u062a \u0645\u0641\u062a\u0627\u062d\u064a\u0629 \u0623\u062e\u0631\u0649.',
        }.get(lang, 'No results found.')
    
    types = {}
    for item in items:
        types.setdefault(item['type'], []).append(item)
    
    parts = []
    total = len(items)
    labels = {
        'project': 'Projets' if lang == 'fr' else 'Projects',
        'stakeholder': 'Organisations' if lang == 'fr' else 'Organizations',
        'resource': 'Ressources' if lang == 'fr' else 'Resources',
    }
    
    if lang == 'fr':
        parts.append(f"{total} resultat{'s' if total>1 else ''} trouve{'s' if total>1 else ''}:")
    else:
        parts.append(f"{total} result{'s' if total!=1 else ''} found:")
    
    for t, entries in types.items():
        parts.append(f"\n**{labels.get(t, t.capitalize())}:**")
        for e in entries[:5]:
            ctx = [x for x in [e.get('country'), e.get('sector')] if x]
            suffix = f" ({', '.join(ctx)})" if ctx else ''
            parts.append(f"  - {e['title']}{suffix}")
        if len(entries) > 5:
            parts.append(f"  ... et {len(entries)-5} autre(s)" if lang=='fr' else f"  ... and {len(entries)-5} more")
    
    return '\n'.join(parts)

print('Build response OK')

Build response OK


In [8]:
# ── Orchestrateur principal ──

def keyword_search(session, query, limit=10):
    """Point d'entree principal du chatbot.
    
    Args:
        session: SQLAlchemy session
        query: question de l'utilisateur
        limit: nombre max de resultats
    Returns:
        dict avec query, parsed, results, response, total_found
    """
    parsed = parse_query(query)
    kw = parsed['keywords']
    country = parsed['country']
    entity = parsed['entity']
    sector = parsed['sector']
    tech = parsed['technology']
    lang = parsed['language']
    
    # Retirer des keywords les mots deja consommes comme filtres
    filter_words_lower = set()
    if country:
        for alias, c in COUNTRY_MAP.items():
            if c == country:
                filter_words_lower.update(alias.split())
    if sector and sector.lower():
        sec_lower = sector.lower()
        for sec, keywords_set in SECTOR_MAP.items():
            if sec.lower() == sec_lower or sec.capitalize() == sector:
                for alias in keywords_set:
                    filter_words_lower.update(alias.split())
    if tech:
        for alias, t in TECH_MAP.items():
            if t == tech:
                filter_words_lower.update(alias.split())
    kw = [w for w in kw if w not in filter_words_lower]
    parsed['keywords'] = kw
    
    print(f'[Search] keywords={kw} country={country} entity={entity} sector={sector} tech={tech}')
    
    all_r = []
    if entity is None or entity == 'project':
        all_r.extend(search_projects(session, kw, country, sector, tech, limit))
    if entity is None or entity == 'stakeholder':
        all_r.extend(search_stakeholders(session, kw, country, limit))
    if entity is None or entity == 'resource':
        all_r.extend(search_resources(session, kw, limit))
    
    # Tri par score descendant
    all_r.sort(key=lambda x: x['score'], reverse=True)
    
    # Deduplication
    seen = set()
    unique = []
    for r in all_r:
        key = (r['type'], r['id'])
        if key not in seen:
            seen.add(key)
            unique.append(r)
    
    return {
        'query': query,
        'parsed': parsed,
        'results': unique[:limit],
        'response': build_response(unique[:limit], lang),
        'total_found': len(unique),
    }

print('Orchestrateur pret')

Orchestrateur pret


---
## 3. Test du chatbot

Posez des questions au chatbot et observez les reponses.

In [9]:
# ── Execution des requetes de test ──

example_queries = [
    'Quels projets IA en sante en Tunisie ?',
    'NLP projects in Morocco',
    'Show me AI startups in Egypt',
    'Machine learning resources',
    'AI healthcare projects',
    'Arabic speech recognition',
    'AI projects in UAE',
]

print('TEST DU KEYWORD SEARCH CHATBOT')
print('='*70)
for q in example_queries:
    print(f"\n{'='*70}")
    print(f'QUESTION: {q}')
    print('='*70)
    start = time.time()
    result = keyword_search(db, q, limit=5)
    elapsed = int((time.time() - start) * 1000)
    print(f'\nREPONSE ({elapsed}ms):')
    print(result['response'])
    if result['results']:
        n = len(result['results'])
        print(f"\nDetails: {n} affiche(s) / {result['total_found']} trouve(s)")
        for r in result['results']:
            print(f"  [{r['type']:12}] score={r['score']:.2f} | {r['title']} ({r['country']})")

TEST DU KEYWORD SEARCH CHATBOT

QUESTION: Quels projets IA en sante en Tunisie ?
[Search] keywords=['ia'] country=Tunisia entity=project sector=Health tech=None

REPONSE (3ms):
Aucun resultat trouve. Essayez avec d'autres mots-cles.

QUESTION: NLP projects in Morocco
[Search] keywords=[] country=Morocco entity=project sector=None tech=NLP

REPONSE (0ms):
No results found. Try different keywords.

QUESTION: Show me AI startups in Egypt
[Search] keywords=['ai'] country=Egypt entity=stakeholder sector=None tech=None

REPONSE (0ms):
2 results found:

**Organizations:**
  - Cairo University AI Lab (Egypt, University)
  - Alexandria University AI Center (Egypt, University)

Details: 2 affiche(s) / 2 trouve(s)
  [stakeholder ] score=1.00 | Cairo University AI Lab (Egypt)
  [stakeholder ] score=1.00 | Alexandria University AI Center (Egypt)

QUESTION: Machine learning resources
[Search] keywords=[] country=None entity=resource sector=Education tech=Machine Learning

REPONSE (0ms):
No results f

---
## 4. Evaluation du Modele 1

Calcule Precision@k, Recall@k, F1@k sur un test set.

**Instructions :**
1. Remplissez `test_set` avec des questions et les IDs attendus
2. Executez la cellule pour evaluer le modele
3. Notez les scores pour comparer avec les Modeles 2 et 3

In [10]:
# ── Test set ──
# Remplacez les IDs par ceux de votre base de donnees
# Format: (question, {(type, id), ...})

test_set = [
    ('AI healthcare projects', {
        ('project', 1),   # AI Diagnostic System
        ('project', 49),  # Tunisia Health AI Platform (a verifier)
    }),
    ('NLP projects in Morocco', {
        # A completer
    }),
    ('Show me AI startups in Egypt', {
        ('stakeholder', 2),   # Cairo University AI Lab
        ('stakeholder', 10),  # Alexandria University AI Center
    }),
    ('Arabic speech recognition', {
        ('project', 6),   # Arabic Speech Recognition
    }),
]

def evaluate(search_fn, test_set, k=10):
    """Evalue un modele sur un test set."""
    results = []
    for query, expected in test_set:
        if not expected:
            continue
        result = search_fn(db, query, limit=k)
        retrieved = set((r['type'], r['id']) for r in result['results'])
        
        tp = len(retrieved & expected)
        fp = len(retrieved - expected)
        fn = len(expected - retrieved)
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        
        results.append({
            'query': query[:50],
            'precision': round(p, 3),
            'recall': round(r, 3),
            'f1': round(f1, 3),
            'retrieved': len(retrieved),
            'expected': len(expected),
            'tp': tp, 'fp': fp, 'fn': fn,
        })
    return results

# Evaluation
results = evaluate(keyword_search, test_set, k=10)
if results:
    print(f"{'Query':<50} {'P':<8} {'R':<8} {'F1':<8} {'Ret':<6} {'Exp':<6}")
    print('-'*86)
    for r in results:
        print(f"{r['query']:<50} {r['precision']:<8.3f} {r['recall']:<8.3f} {r['f1']:<8.3f} {r['retrieved']:<6} {r['expected']:<6}")
    avg_p = sum(r['precision'] for r in results) / len(results)
    avg_r = sum(r['recall'] for r in results) / len(results)
    avg_f = sum(r['f1'] for r in results) / len(results)
    print(f"\n{'MOYENNE':<50} {avg_p:<8.3f} {avg_r:<8.3f} {avg_f:<8.3f}")
else:
    print('Test set vide ou sans expected IDs. Completez test_set avec vos IDs.')

[Search] keywords=['ai'] country=None entity=project sector=Health tech=None
[Search] keywords=['ai'] country=Egypt entity=stakeholder sector=None tech=None
[Search] keywords=['arabic'] country=None entity=None sector=None tech=Speech Recognition
Query                                              P        R        F1       Ret    Exp   
--------------------------------------------------------------------------------------
AI healthcare projects                             1.000    0.500    0.667    1      2     
Show me AI startups in Egypt                       1.000    1.000    1.000    2      2     
Arabic speech recognition                          0.500    1.000    0.667    2      1     

MOYENNE                                            0.833    0.833    0.778   


---
## 5. Comparaison entre Modeles

*(A remplir quand les Modeles 2 et 3 sont prets)*

| Modele | Precision@10 | Recall@10 | F1@10 | Temps moyen |
|--------|:-----------:|:--------:|:-----:|:----------:|
| **1. Keyword Search** | | | | |
| **2. Semantic Search** | | | | |
| **3. RAG** | | | | |

**Le meilleur modele** sera celui avec le **F1-score** le plus eleve et le **temps de reponse** le plus bas.

In [11]:
# ── Comparaison finale des 3 modeles ──
# Utilisation:
# models = {
#     'Keyword Search': keyword_search,  # Ce modele
#     'Semantic Search': semantic_fn,    # Modele 2
#     'RAG': rag_fn,                      # Modele 3
# }
# compare(models, test_set)

def compare(models: dict, test_set, k=10):
    """Compare plusieurs modeles sur un meme test set."""
    print(f"{'Modele':<20} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Temps(ms)':<12}")
    print('-'*68)
    
    for name, fn in models.items():
        precisions, recalls, f1s, times = [], [], [], []
        for query, expected in test_set:
            if not expected:
                continue
            start = time.time()
            result = fn(db, query, limit=k)
            elapsed = int((time.time() - start) * 1000)
            
            retrieved = set((r['type'], r['id']) for r in result['results'])
            tp = len(retrieved & expected)
            fp = len(retrieved - expected)
            fn_c = len(expected - retrieved)
            
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn_c) if (tp + fn_c) > 0 else 0.0
            f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
            
            precisions.append(p)
            recalls.append(r)
            f1s.append(f)
            times.append(elapsed)
        
        avg_p = sum(precisions) / len(precisions) if precisions else 0
        avg_r = sum(recalls) / len(recalls) if recalls else 0
        avg_f = sum(f1s) / len(f1s) if f1s else 0
        avg_t = sum(times) / len(times) if times else 0
        
        print(f"{name:<20} {avg_p:<12.3f} {avg_r:<12.3f} {avg_f:<12.3f} {avg_t:<12.0f}")
    
    print('\nLe meilleur modele est celui avec le F1 le plus eleve.')

print('Fonction de comparaison prete.')
print('Exemple d\'utilisation:')
print('  models = {"Keyword Search": keyword_search}')
print('  compare(models, test_set)')

Fonction de comparaison prete.
Exemple d'utilisation:
  models = {"Keyword Search": keyword_search}
  compare(models, test_set)


---
## 5b. Evaluation approfondie

Calcule toutes les metriques sur un test set :
- **Accuracy, Precision, Recall, F1-Score** (retrieval)
- **ROUGE-L, BLEU** (similarite textuelle de la reponse)
- **Response Time** (temps de reponse)
- **Semantic Similarity** (similarite semantique query↔reponse)

*Note : ROUGE-L et BLEU comparent la reponse du chatbot a une reponse de reference attendue (`expected_response` dans le test set).*

**Instructions :**
1. Completez `test_set_advanced` avec vos questions, IDs attendus et reponse attendue
2. Executez la cellule

In [12]:
# ── Evaluation approfondie ──

import math
from collections import Counter

# ── Test set avec reponses attendues ──
# Format: (question, {(type, id), ...}, "expected_response_text")
# expected_response peut etre vide "" si non disponible

test_set_advanced = [
    ('AI healthcare projects', {
        ('project', 1),   # AI Diagnostic System
    }, "1 result found:\n\n**Projects:**\n  - AI Diagnostic System (Egypt, Health)"),
    ('Show me AI startups in Egypt', {
        ('stakeholder', 2),   # Cairo University AI Lab
        ('stakeholder', 10),  # Alexandria University AI Center
    }, "2 results found:\n\n**Organizations:**\n  - Cairo University AI Lab (Egypt, University)\n  - Alexandria University AI Center (Egypt, University)"),
    ('Arabic speech recognition', {
        ('project', 6),   # Arabic Speech Recognition
    }, ""),
]


# ── Fonctions metriques ──

def rouge_l_score(reference, hypothesis):
    """ROUGE-L: F1 du LCS entre reference et hypothese."""
    if not reference or not hypothesis:
        return 0.0
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    m, n = len(ref_tokens), len(hyp_tokens)
    # LCS length
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    if lcs == 0:
        return 0.0
    prec = lcs / n
    rec = lcs / m
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return round(f1, 4)


def bleu_score(reference, hypothesis, max_n=4):
    """BLEU simplifie (unigrammes a 4-grammes) sans brevity penalty."""
    if not reference or not hypothesis:
        return 0.0
    ref_tokens = reference.lower().split()
    hyp_tokens = hypothesis.lower().split()
    if len(hyp_tokens) == 0:
        return 0.0
    
    precisions = []
    for n in range(1, min(max_n, len(ref_tokens), len(hyp_tokens)) + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens)-n+1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens)-n+1))
        match_count = sum(min(count, ref_ngrams.get(ngram, 0)) for ngram, count in hyp_ngrams.items())
        total_count = max(1, len(list(hyp_ngrams.elements())))
        precisions.append(match_count / total_count if total_count > 0 else 0.0)
    
    if not precisions:
        return 0.0
    
    # Brevity penalty
    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens))) if len(hyp_tokens) > 0 else 0.0
    
    geo_mean = math.exp(sum(math.log(max(p, 1e-10)) for p in precisions) / len(precisions))
    return round(bp * geo_mean, 4)


def eval_advanced(search_fn, test_set, k=10):
    """Evaluate un modele avec toutes les metriques."""
    results = []
    semantic_ok = False
    try:
        from sentence_transformers import SentenceTransformer
        sem_model = SentenceTransformer('all-MiniLM-L6-v2')
        semantic_ok = True
    except Exception:
        pass
    
    for row in test_set:
        if len(row) == 3:
            query, expected_ids, expected_response = row
        else:
            query, expected_ids = row
            expected_response = ""
        
        if not expected_ids:
            continue
        
        start = time.time()
        result = search_fn(db, query, limit=k)
        elapsed = int((time.time() - start) * 1000)
        
        retrieved = set((r['type'], r['id']) for r in result['results'])
        response_text = result['response']
        
        tp = len(retrieved & expected_ids)
        fp = len(retrieved - expected_ids)
        fn = len(expected_ids - retrieved)
        
        n_total_expected = len(expected_ids)
        n_retrieved = len(retrieved)
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        
        # Accuracy: (TP+TN)/(TP+TN+FP+FN)
        # TN = total_possible - n_expected - (n_retrieved - tp)
        # Simplified: TN is undefined in retrieval, so we omit it
        acc = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        
        # ROUGE-L & BLEU
        rl = rouge_l_score(expected_response, response_text) if expected_response else None
        bl = bleu_score(expected_response, response_text) if expected_response else None
        
        # Semantic Similarity
        ss = None
        if semantic_ok and response_text:
            emb_q = sem_model.encode(query)
            emb_r = sem_model.encode(response_text)
            from numpy import dot
            from numpy.linalg import norm
            ss = round(float(dot(emb_q, emb_r) / (norm(emb_q) * norm(emb_r) + 1e-10)), 4)
        
        results.append({
            'query': query[:50],
            'accuracy': round(acc, 3),
            'precision': round(p, 3),
            'recall': round(r, 3),
            'f1': round(f1, 3),
            'rouge_l': rl,
            'bleu': bl,
            'sem_sim': ss,
            'time_ms': elapsed,
            'retrieved': n_retrieved,
            'expected': n_total_expected,
            'tp': tp, 'fp': fp, 'fn': fn,
        })
    
    return results, semantic_ok


# ── Execution ──
print('='*100)
print('EVALUATION APPROFONDIE DU MODELE 1 (KEYWORD SEARCH)')
print('='*100)

adv_results, sem_ok = eval_advanced(keyword_search, test_set_advanced, k=10)

if not adv_results:
    print('\nTest set vide ou sans expected IDs.')
else:
    # En-tete
    headers = ['Query', 'Acc', 'P', 'R', 'F1', 'R-L', 'BLEU', 'SemSim', 'Time(ms)']
    print(f"{'Query':<45} {'Acc':<8} {'P':<8} {'R':<8} {'F1':<8} {'R-L':<8} {'BLEU':<8} {'SemSim':<8} {'Time':<8}")
    print('-'*105)
    
    totals = {'acc': [], 'p': [], 'r': [], 'f1': [], 'rl': [], 'bl': [], 'ss': [], 't': []}
    
    for r in adv_results:
        rl_str = f"{r['rouge_l']:.3f}" if r['rouge_l'] is not None else 'N/A'
        bl_str = f"{r['bleu']:.3f}" if r['bleu'] is not None else 'N/A'
        ss_str = f"{r['sem_sim']:.3f}" if r['sem_sim'] is not None else 'N/A'
        print(f"{r['query']:<45} {r['accuracy']:<8.3f} {r['precision']:<8.3f} {r['recall']:<8.3f} {r['f1']:<8.3f} {rl_str:<8} {bl_str:<8} {ss_str:<8} {r['time_ms']:<8}")
        totals['acc'].append(r['accuracy'])
        totals['p'].append(r['precision'])
        totals['r'].append(r['recall'])
        totals['f1'].append(r['f1'])
        if r['rouge_l'] is not None: totals['rl'].append(r['rouge_l'])
        if r['bleu'] is not None: totals['bl'].append(r['bleu'])
        if r['sem_sim'] is not None: totals['ss'].append(r['sem_sim'])
        totals['t'].append(r['time_ms'])
    
    # Moyennes
    avg = lambda lst: sum(lst)/len(lst) if lst else 0.0
    rl_avg = f"{avg(totals['rl']):.3f}" if totals['rl'] else 'N/A'
    bl_avg = f"{avg(totals['bl']):.3f}" if totals['bl'] else 'N/A'
    ss_avg = f"{avg(totals['ss']):.3f}" if totals['ss'] else 'N/A'
    print('-'*105)
    print(f"{'MOYENNE':<45} {avg(totals['acc']):<8.3f} {avg(totals['p']):<8.3f} {avg(totals['r']):<8.3f} {avg(totals['f1']):<8.3f} {rl_avg:<8} {bl_avg:<8} {ss_avg:<8} {avg(totals['t']):<8.0f}")

if sem_ok:
    print('\n[Semantic Similarity] OK (sentence-transformers)')
else:
    print('\n[Semantic Similarity] Non disponible. Installez: pip install sentence-transformers')


EVALUATION APPROFONDIE DU MODELE 1 (KEYWORD SEARCH)

[Search] keywords=['ai'] country=None entity=project sector=Health tech=None
[Search] keywords=['ai'] country=Egypt entity=stakeholder sector=None tech=None
[Search] keywords=['arabic'] country=None entity=None sector=None tech=Speech Recognition
Query                                         Acc      P        R        F1       R-L      BLEU     SemSim   Time    
---------------------------------------------------------------------------------------------------------
AI healthcare projects                        1.000    1.000    1.000    1.000    1.000    1.000    0.678    2       
Show me AI startups in Egypt                  1.000    1.000    1.000    1.000    1.000    1.000    0.559    0       
Arabic speech recognition                     0.500    0.500    1.000    0.667    N/A      N/A      0.780    0       
---------------------------------------------------------------------------------------------------------
MOYENNE         

---
## 6. Chat interactif

Posez vos questions en continu. Tapez `quit`, `exit` ou `q` pour quitter.

In [13]:
print("=== KEYWORD SEARCH CHATBOT (Modele 1) ===")
print("Tapez 'quit', 'exit' ou 'q' pour quitter.\n")

while True:
    q = input("\nVotre question: ").strip()
    if q.lower() in ('quit', 'exit', 'q'):
        print("Au revoir!")
        break
    if not q:
        continue

    start = time.time()
    result = keyword_search(db, q, limit=5)
    elapsed = int((time.time() - start) * 1000)

    print(f"\nTemps: {elapsed}ms")
    print(f"Reponse: {result['response']}")
    if result['results']:
        p = result['parsed']
        print(f"Parsed: keywords={result['parsed']['keywords']} pays={p['country']} type={p['entity']} secteur={p['sector']} tech={p['technology']}")
        print(f"Resultats: {len(result['results'])} affiche(s) / {result['total_found']} trouve(s)")
        for r in result['results']:
            print(f"  [{r['type']:12}] score={r['score']:.2f} | {r['title']} ({r['country']})")
    print('-' * 60)


=== KEYWORD SEARCH CHATBOT (Modele 1) ===
Tapez 'quit', 'exit' ou 'q' pour quitter.

[Search] keywords=[] country=Tunisia entity=project sector=None tech=None

Temps: 0ms
Reponse: 1 result found:

**Projects:**
  - Smart Waste Management (Tunisia, Environment)
Parsed: keywords=[] pays=Tunisia type=project secteur=None tech=None
Resultats: 1 affiche(s) / 1 trouve(s)
  [project     ] score=1.00 | Smart Waste Management (Tunisia)
------------------------------------------------------------
[Search] keywords=[] country=Tunisia entity=stakeholder sector=None tech=None

Temps: 163ms
Reponse: 1 result found:

**Organizations:**
  - Tunisian AI Hub (Tunisia, Startup)
Parsed: keywords=[] pays=Tunisia type=stakeholder secteur=None tech=None
Resultats: 1 affiche(s) / 1 trouve(s)
  [stakeholder ] score=0.50 | Tunisian AI Hub (Tunisia)
------------------------------------------------------------
[Search] keywords=['ffrance'] country=None entity=stakeholder sector=None tech=None

Temps: 0ms
Reponse:

---
## 7. Fermeture de la connexion

In [12]:
db.close()
print('Connexion DB fermee.')

Connexion DB fermee.
